In [268]:
import warnings
warnings.filterwarnings('ignore')

In [269]:
import pandas as pd
import numpy as np
import pyreadr
import joblib
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

from dash import Dash, html, dcc, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

In [270]:
pd.set_option('display.max_columns', 100)

In [271]:
dem_uncont = 34
rep_uncont = 31

In [272]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont)
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims

,seats,winner
0,54,Democrats
1,55,Democrats
2,49,Republicans
3,54,Democrats
4,48,Republicans
...,...,...
7995,51,Democrats
7996,51,Democrats
7997,53,Democrats
7998,51,Democrats


In [273]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 51 else (100-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,36,1,0.0125,Republicans
1,38,1,0.0125,Republicans
2,40,2,0.0250,Republicans
3,41,4,0.0500,Republicans
4,42,8,0.1000,Republicans


In [274]:
np.unique(seat_sims['seats']).shape[0]

27

In [275]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'}, template='plotly_white')
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=50.5, line_width=1, line_color='black', annotation_text='51 seats required for Dem majority', 
                    annotation_position='top right')
sims_hist

In [276]:
output_ts = pd.read_csv('model_output/output_over_time.csv')
output_ts.head()

,date,y,geo,type
0,2026-09-27,51.348375,US Senate,seats
1,2026-09-27,58.600000,US Senate,chance
2,2026-09-27,3.816193,US Senate,seats_sd
3,2026-09-27,38.771876,AL,y_pred
4,2026-09-27,3.537242,AL,y_pred_sd


In [277]:
chance_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'chance')]
chance_over_time['rep_chance'] = chance_over_time['y'].map(lambda x: 100 - x)
chance_over_time = chance_over_time.rename({'y': 'Democrats', 'rep_chance': 'Republicans'}, axis=1)
chance_time_ser = px.line(chance_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
chance_time_ser.update_traces(hovertemplate="%{y:.1f}%")
chance_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Win Probability (%)',
    title=dict(text="Senate Win Probability Over Time"),
    hovermode="x",
    showlegend=False
)
chance_time_ser

In [278]:
seats_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'seats')]
seats_over_time['rep_seats'] = seats_over_time['y'].map(lambda x: 100 - x)
seats_over_time = seats_over_time.rename({'y': 'Democrats', 'rep_seats': 'Republicans'}, axis=1)
seats_time_ser = px.line(seats_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                        color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
seats_time_ser.update_traces(hovertemplate="%{y:.1f}")
seats_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Average Seats Over All Simulations',
    title=dict(text="Projected Seats Over Time"),
    hovermode="x",
    showlegend=False
)
seats_time_ser

In [279]:
joblib.dump(sims_hist, 'display_data/sims_histogram.pkl')
joblib.dump(chance_time_ser, 'display_data/chance_time_ser.pkl')
joblib.dump(seats_time_ser, 'display_data/seats_time_ser.pkl')

['display_data/seats_time_ser.pkl']

In [280]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [281]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,7950,7951,7952,7953,7954,7955,7956,7957,7958,7959,7960,7961,7962,7963,7964,7965,7966,7967,7968,7969,7970,7971,7972,7973,7974,7975,7976,7977,7978,7979,7980,7981,7982,7983,7984,7985,7986,7987,7988,7989,7990,7991,7992,7993,7994,7995,7996,7997,7998,7999
AL,-5.198238,-11.707929,-9.664246,-9.919610,-7.728281,-11.221533,-9.706824,-3.475918,-9.998444,-15.406475,1.075986,-11.023454,-8.807472,-9.444102,-10.148542,-7.529956,-7.749207,-10.306445,-7.891548,-5.354373,-10.630633,-6.954544,-4.954244,-7.192615,-6.680626,-12.513929,-11.972506,-5.941405,-10.988575,-11.138037,-8.530512,-9.728320,-9.436854,-5.962681,-14.054882,-13.038151,-5.833098,-8.876433,-6.566120,-10.619791,-8.521858,-6.750210,-11.835108,-8.681818,-10.837195,-11.390125,-10.894414,-12.968015,-5.904654,-9.451213,...,-9.615333,-8.979590,-7.799350,-13.067032,-5.043899,-9.683889,-10.802252,-5.686383,-10.162556,-6.185727,-14.199407,-14.202705,-8.468361,-5.137191,-15.479804,-5.075354,-14.430837,-0.985766,-9.935884,-6.818503,-9.067942,-7.727564,-2.623460,-19.254743,-1.272806,-11.848972,-9.476969,-11.475504,-5.764978,-10.474831,-8.730706,-10.764110,-8.881666,-5.952152,-14.706011,-10.291438,-8.909735,-16.731431,-7.933448,-8.004576,-8.869358,-12.110969,-8.559978,-8.454528,-10.112967,-10.983884,-12.973261,-7.862883,-10.644304,-10.149447
AK,0.555077,5.721794,-0.109608,5.620769,-0.994800,4.636022,4.978870,5.202950,-0.582709,-5.637400,10.937698,0.288170,-1.029639,6.546964,-0.698753,3.196909,5.740657,4.694392,-0.320131,2.878924,6.930264,4.212693,2.405779,5.127884,4.959295,1.471032,2.461823,3.675923,-0.893704,1.584592,3.464005,5.770212,7.028874,8.298375,2.837687,-0.266600,3.703851,0.059762,-1.027430,3.345790,-1.264437,3.737289,7.029228,4.092456,2.670272,-1.713151,0.965411,3.051616,2.265424,4.448577,...,3.933035,2.935036,10.035352,-0.255269,7.565347,2.724310,1.248549,3.587977,-1.491441,10.521264,2.442643,-0.872001,7.050896,3.323654,3.767373,1.614502,4.782581,2.230196,-0.603842,7.408141,1.738430,3.410024,8.364981,0.173873,3.945553,-2.329206,6.435643,4.919201,3.766112,0.847581,-3.129727,5.221503,-0.687934,5.989110,1.722591,3.023514,3.231980,7.183082,4.044586,4.062037,0.047306,4.199971,3.904777,0.328458,3.948569,0.938287,1.728715,0.664984,3.579697,1.755137
AR,-10.635485,-12.514538,-8.895800,-11.619024,-10.328748,-8.514844,-9.609939,-6.144544,-14.592400,-14.622733,-1.587214,-12.230196,-13.130463,-7.746446,-12.612566,-11.169217,-10.515896,-9.001726,-11.295659,-7.722151,-11.748292,-11.263041,-7.496636,-7.743224,-11.989036,-9.237727,-11.727195,-8.420768,-11.714766,-11.522357,-8.889293,-9.597902,-7.083927,-7.860643,-13.907282,-13.322508,-6.753931,-10.148716,-11.707063,-10.784251,-10.756339,-6.782927,-12.708472,-14.341699,-13.050746,-11.269232,-9.456183,-14.415122,-11.113180,-12.376275,...,-13.323265,-11.749190,-8.431744,-13.989798,-4.215336,-12.667369,-13.270866,-11.940048,-10.662561,-4.514418,-15.020208,-12.585886,-8.621628,-8.059624,-13.912418,-5.145022,-15.918846,-1.667448,-9.728535,-8.872825,-12.179990,-6.771549,-5.158263,-15.980676,-3.650673,-16.958977,-9.681760,-16.429099,-5.760763,-10.136797,-9.585333,-11.212760,-10.115602,-3.013967,-17.829096,-10.148451,-8.200988,-12.839037,-6.397509,-12.626807,-11.535261,-11.110963,-9.130814,-7.397268,-11.571041,-13.986366,-17.522224,-11.731674,-7.998409,-11.177974
CO,10.336929,11.435589,6.473932,13.296042,8.243368,11.712268,13.518096,12.496994,9.525498,6.723940,12.002784,12.695270,9.651916,15.243394,7.204961,15.022099,12.835938,10.882977,5.703450,5.043255,11.932079,17.854883,9.019716,15.352541,12.659959,10.676694,10.597446,19.018474,6.743555,12.854699,13.152788,9.480954,8.415610,9.425290,10.923565,5.046153,15.404961,10.231275,14.435508,8.440231,12.023523,10.975247,10.878914,12.773057,9.531376,10.712654,6.797927,10.188156,11.632529,8.737502,...,12.937549,7.683936,16.371780,9.377217,12.269736,6.134845,10.680841,10.5706

In [282]:
sim_corr = post_untransp.corr()

In [283]:
sim_corr.head()

,AL,AK,AR,CO,DE,FL,GA,ID,IL,IA,KS,KY,LA,ME,MA,MI,MN,MS,MT,NE,NH,NJ,NM,NC,OH,OK,OR,RI,SC,SD,TN,TX,VA,WV,WY
AL,1.000000,0.515695,0.738772,0.487240,0.497142,0.699083,0.529828,0.735555,0.704726,0.739027,0.731938,0.540768,0.749456,0.731500,0.519478,0.729029,0.704835,0.742914,0.694016,0.720549,0.498000,0.506527,0.505234,0.702829,0.503941,0.756859,0.507075,0.487558,0.488181,0.512184,0.556274,0.693996,0.504951,0.739365,0.737405
AK,0.515695,1.000000,0.518535,0.524462,0.525597,0.508579,0.557373,0.526887,0.504450,0.522711,0.522217,0.568409,0.525755,0.523837,0.560607,0.513539,0.498497,0.518300,0.496384,0.518301,0.561833,0.539803,0.553870,0.498798,0.554912,0.522600,0.547869,0.549337,0.539483,0.543263,0.598262,0.507940,0.544452,0.521338,0.520718
AR,0.738772,0.518535,1.000000,0.486962,0.499848,0.703739,0.527019,0.737910,0.699261,0.740893,0.736584,0.537463,0.742050,0.732341,0.522543,0.728378,0.699035,0.726015,0.693602,0.720586,0.508340,0.500897,0.501666,0.704530,0.504907,0.744104,0.501891,0.494754,0.495301,0.514553,0.556796,0.699023,0.499168,0.735688,0.734959
CO,0.487240,0.524462,0.486962,1.000000,0.709019,0.653283,0.554219,0.498178,0.656061,0.503280,0.487579,0.549489,0.504258,0.487846,0.539459,0.489438,0.644107,0.493538,0.654649,0.476982,0.714105,0.715060,0.722793,0.650014,0.705195,0.499633,0.710588,0.704601,0.701045,0.528987,0.573583,0.657204,0.704302,0.497623,0.494468
DE,0.497142,0.525597,0.499848,0.709019,1.000000,0.665520,0.543313,0.504492,0.664581,0.507025,0.498133,0.547891,0.503238,0.501665,0.548410,0.501136,0.663479,0.499345,0.663008,0.487458,0.717941,0.725918,0.728686,0.661061,0.714137,0.515787,0.710004,0.711666,0.707794,0.519550,0.577994,0.657940,0.719028,0.499519,0.509416


In [284]:
post.shape

(35, 8000)

In [285]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 51 - dem_uncont: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (51 - dem_uncont)
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (100 - seats_won_dem - dem_uncont - rep_uncont) - (50 - rep_uncont)
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [286]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:06<00:00, 1245.46it/s]


array(['NE', 'ME', 'AK', ..., 'AK', 'RI', 'AK'],
      shape=(8000,), dtype='<U32')

In [287]:
data = pd.read_csv('../../model_output/senate_predictions.csv')
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.843901,0,1,39.806386,1.123208,52.528412,1.123208,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.123208,1.059815,12.722026,-21.777142,2.405409,-1,4.810817,39.747542,3.571036,0.2125,3,32.619734,46.692898
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.843901,1,0,56.276368,0.897376,38.855274,0.897376,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.897376,0.947299,-17.421094,20.773037,48.015860,1,96.031719,60.846591,3.500816,99.9250,4,54.057091,67.718643
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.843901,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.866290,47.316437,1,94.632875,63.269649,3.504418,99.9875,5,56.432178,70.223397


In [288]:
data['tipping_point_prob'] = data['state_po'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.843901,0,1,39.806386,1.123208,52.528412,1.123208,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.123208,1.059815,12.722026,-21.777142,2.405409,-1,4.810817,39.747542,3.571036,0.2125,3,32.619734,46.692898,0.0000
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.843901,1,0,56.276368,0.897376,38.855274,0.897376,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.897376,0.947299,-17.421094,20.773037,48.015860,1,96.031719,60.846591,3.500816,99.9250,4,54.057091,67.718643,0.0500
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.843901,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.866290,47.316437,1,94.632875,63.269649,3.504418,99.9875,5,56.432178,70.223397,0.2375


In [289]:
data.sort_values('tipping_point_prob', ascending=False).head(5)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
15,15,Michigan,Abdul El-Sayed,Mike J. Rogers,False,False,D,MI,"EL-SAYED, ABDUL","ROGERS, MICHAEL J",4.719966e+07,15432313.28,6.263198e+07,75.360329,24.639671,-0.192456,49.278763,77.709102,2.333008,4.148895,12.968600,0.319413,32.445883,7404258,2673073,10077331,73.474395,26.525605,0.987401,-8.843901,0,0,49.401572,11.422765,45.486166,11.422765,East North Central,2026,0.0,0.00000,0.00000,2,Senate,5679.179195,11.422765,3.379758,-3.915406,8.458989,25.360329,0,50.720658,50.804854,3.541156,58.8375,16,44.022678,57.823249,10.8875
13,13,Maine,Troy Jackson,Susan Collins,False,True,R,ME,"JACKSON, TROY DALE","COLLINS, SUSAN M.",2.397712e+07,15299978.61,3.927710e+07,61.046056,38.953944,3.820912,53.544306,94.249770,0.897865,1.689465,0.930085,0.342309,36.189622,526309,836050,1362359,38.632181,61.367819,0.987401,-8.843901,0,1,51.137801,8.960811,46.001893,8.960811,New England,2026,0.0,0.00000,0.00000,3,Senate,3726.620896,8.960811,2.993461,-5.135909,16.485725,11.046056,-1,22.092111,50.697039,3.582988,57.6250,14,43.658521,57.683332,10.2875
31,31,Texas,James Talarico,Ken Paxton,False,False,R,TX,"TALARICO, JAMES","PAXTON, WARREN KENNETH JR.",1.397766e+08,20997954.85,1.607746e+08,86.939504,13.060496,-5.916874,43.060966,48.286718,4.530331,31.642337,13.173480,0.180615,33.823334,24400697,4744808,29145505,83.720275,16.279725,0.987401,-8.843901,0,0,49.934912,14.319705,46.036130,14.319705,West South Central,2026,0.0,0.25284,-0.25284,6,Senate,7558.477337,14.319705,3.784139,-3.898782,-2.989847,36.939504,0,73.879008,52.023768,3.649103,71.2250,32,44.889553,59.282126,9.4875
10,10,Kansas,Adam Hamilton,Roger Marshall,False,True,R,KS,"HAMILTON, ADAM","MARSHALL, ROGER W",1.652601e+07,2514153.97,1.904016e+07,86.795522,13.204478,-8.032578,41.794263,80.203805,2.235100,8.765322,5.263831,0.443291,35.603269,2124059,813821,2937880,72.299039,27.700961,0.987401,-8.843901,0,1,49.320108,5.626859,47.130519,5.626859,West North Central,2026,0.0,0.00000,0.00000,2,Senate,7533.462650,5.626859,2.372100,-2.189589,-7.221255,36.795522,-1,73.591044,49.699138,3.536260,46.2625,11,42.777972,56.731228,8.6500
28,28,South Carolina,Annie Andrews,Darline Graham,False,False,R,SC,"ANDREWS, ANNIE","GRAHAM, DARLINE",2.356337e+07,724282.27,2.428766e+07,97.017900,2.982100,-8.282832,40.938317,67.144996,1.378347,4.124393,24.827438,0.196629,32.129153,3477869,1640556,5118425,67.948031,32.051969,0.987401,-8.843901,0,0,46.412972,3.336880,45.948020,3.336880,South Atlantic,2026,0.0,0.00000,0.00000,5,Senate,9412.472854,3.336880,1.826713,-0.464953,-7.721764,47.017900,0,94.035799,49.948941,3.554796,49.3875,29,42.915577,56.913169,8.1625


In [290]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand, dem_inc_any, rep_inc_any):
    indie_d = ['Dan Osborn', 'Brian Bengs', 'Todd Achilles', 'Seth Bodnar']
    indie_r = []
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (I)'
        dem_color = '#792ba6'
    else:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (D)'
        dem_color = '#366bbf'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (I)'
        rep_color = '#792ba6'
    else:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (R)'
        rep_color = '#e63929'

    return f'<b style="color:{dem_color};">' + dem_lab + f'</b> vs <b style="color:{rep_color};">' + rep_lab + '</b>'

In [291]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                          x['dem_inc_any'], x['rep_inc_any']), axis=1)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b..."
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>..."
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.843901,0,1,39.806386,1.123208,52.528412,1.123208,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.123208,1.059815,12.722026,-21.777142,2.405409,-1,4.810817,39.747542,3.571036,0.2125,3,32.619734,46.692898,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<..."
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.843901,1,0,56.276368,0.897376,38.855274,0.897376,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.897376,0.947299,-17.421094,20.773037,48.015860,1,96.031719,60.846591,3.500816,99.9250,4,54.057091,67.718643,0.0500,Safe D,"<b style=""color:#366bbf;"">John Hickenlooper* (..."
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.843901,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.866290,47.316437,1,94.632875,63.269649,3.504418,99.9875,5,56.432178,70.223397,0.2375,Safe D,"<b style=""color:#366bbf;"">Chris Coons* (D)</b>..."


In [292]:
data['projected_winner'] = data['chance'].map(lambda x: 'D' if x > 50 else 'R')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.843901,0,1,39.806386,1.123208,52.528412,1.123208,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.123208,1.059815,12.722026,-21.777142,2.405409,-1,4.810817,39.747542,3.571036,0.2125,3,32.619734,46.692898,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R


In [293]:
data['hold'] = data['projected_winner'] ==  data['current_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip


In [294]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+17.9
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.843901,0,1,39.806386,1.123208,52.528412,1.123208,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.123208,1.059815,12.722026,-21.777142,2.405409,-1,4.810817,39.747542,3.571036,0.2125,3,32.619734,46.692898,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.5


In [295]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+17.9,99.4000,0.6,99.4,<1%,>99%
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4,17.5125,82.5,17.5,82.5%,17.5%
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.843901,0,1,39.806386,1.123208,52.528412,1.123208,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.123208,1.059815,12.722026,-21.777142,2.405409,-1,4.810817,39.747542,3.571036,0.2125,3,32.619734,46.692898,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.5,99.7875,0.2,99.8,<1%,>99%


In [296]:
data['swing_24_to_26'] = data['y_pred'].astype(float).map(lambda x: x - (100 - x)) - data['dem_2p_24'].astype(float).map(lambda x: x - (100 - x))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+17.9,99.4000,0.6,99.4,<1%,>99%,12.993187
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4,17.5125,82.5,17.5,82.5%,17.5%,20.059961
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.843901,0,1,39.806386,1.123208,52.528412,1.123208,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.123208,1.059815,12.722026,-21.777142,2.405409,-1,4.810817,39.747542,3.571036,0.2125,3,32.619734,46.692898,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.5,99.7875,0.2,99.8,<1%,>99%,10.835071


In [297]:
data['disp_24_to_26_swing'] = data['swing_24_to_26'].map(lambda x: f'D+{x:.1f}' if x > 0 else f'R+{abs(x):.1f}')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26,disp_24_to_26_swing
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.843901,0,0,38.246192,1.759568,49.145126,1.759568,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.759568,1.326487,10.898934,-20.782541,-48.013273,0,-96.026546,41.055982,3.546625,0.6000,1,34.205004,48.172040,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+17.9,99.4000,0.6,99.4,<1%,>99%,12.993187,D+13.0
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.843901,0,1,49.424854,4.570436,48.164401,4.570436,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,4.570436,2.137858,-1.260453,-4.066704,35.161107,-1,70.322215,53.183418,3.402916,82.4875,2,46.512543,59.997207,5.7625,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4,17.5125,82.5,17.5,82.5%,17.5%,20.059961,D+20.1


In [298]:
tab_data = data[['state', 'dem_cand', 'rep_cand', 'rating', 'disp_dem_chance', 'disp_rep_chance', 'projected_2p_margin', 'disp_24_to_26_swing',
                 'tipping_point_prob']]
for party in ['dem', 'rep']:
    tab_data[f'disp_{party}_chance'] = tab_data[f'disp_{party}_chance'].map(lambda x: f'<p style="color:{'blue' if party == 'dem' else 'red'};">{x}</p>')
    tab_data[f'{party}_cand'] = tab_data[f'{party}_cand'].map(lambda x: f'{x} (Ind)' if x in ['Dan Osborn', 'Brian Bengs', 
                                                                                              'Todd Achilles', 'Seth Bodnar'] else x)
tab_data = tab_data.rename({
    'state': 'State',
    'dem_cand': 'Democrat',
    'rep_cand': 'Republican',
    'rating': 'Rating',
    'disp_dem_chance': 'Dem Chance',
    'disp_rep_chance': 'Rep Chance',
    'projected_2p_margin': 'Projected Margin',
    'tipping_point_prob': 'Tipping Point Chance',
    'disp_24_to_26_swing': 'Swing from 2024 Pres'
}, axis=1)
tab_data.head()

,State,Democrat,Republican,Rating,Dem Chance,Rep Chance,Projected Margin,Swing from 2024 Pres,Tipping Point Chance
0,Alabama,Everett Weiss,Barry Moore,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+17.9,D+13.0,0.0000
1,Alaska,Mary Peltola,Dan S. Sullivan,Likely D,"<p style=""color:blue;"">82.5%</p>","<p style=""color:red;"">17.5%</p>",D+6.4,D+20.1,5.7625
2,Arkansas,Hallie Shoffner,Tom Cotton,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+20.5,D+10.8,0.0000
3,Colorado,John Hickenlooper,Mark Baisley,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+21.7,D+10.4,0.0500
4,Delaware,Chris Coons,Michael Katz,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+26.5,D+11.6,0.2375


In [299]:
data['rep_pred'] = data['y_pred'].map(lambda x: 100 - x)
data['projected_2p_margin_number'] = data['rep_pred'] - data['y_pred']
data['proj_seat_lean'] = data['projected_2p_margin_number'] - data['generic_ballot_avg']
sv_bias = float(data.sort_values('tipping_point_prob', ascending=False).reset_index().loc[0, 'proj_seat_lean']) # Seats-votes bias, + = R, - = D
sv_bias

7.234192859500766

In [300]:
topline_stats = pd.read_csv('display_data/topline_stats.csv')
if 'sv_bias' in topline_stats['vars'].values:
    topline_stats = topline_stats[topline_stats['vars'] != 'sv_bias']
topline_stats = pd.concat([topline_stats, pd.DataFrame({'vars': ['sv_bias'], 'x': sv_bias})], axis=0)
topline_stats.to_csv('display_data/topline_stats.csv')
topline_stats

,vars,x
0,means_seats_tot,52.197875
1,chamber_win_chance,65.187500
0,sv_bias,7.234193


In [301]:
mean_seats_tot = topline_stats[topline_stats['vars'] == 'means_seats_tot']['x'].values[0]
chamber_win_chance = topline_stats[topline_stats['vars'] == 'chamber_win_chance']['x'].values[0]

In [302]:
chances = topline_stats[topline_stats['vars'] == 'chamber_win_chance'].set_index(['vars']).T
chances['Republicans'] = chances['chamber_win_chance'].map(lambda x: 100 - x)
chances = chances.rename({'chamber_win_chance': 'Democrats'}, axis=1).T.reset_index()
chances = chances.rename({'x': 'Win Probability'}, axis=1)
seats = topline_stats[topline_stats['vars'] == 'means_seats_tot'].set_index(['vars']).T
seats['Republicans'] = seats['means_seats_tot'].map(lambda x: 100 - x)
seats = seats.rename({'means_seats_tot': 'Democrats'}, axis=1).T.reset_index().rename({'x': 'Seat Share'}, axis=1)
summary_stats = pd.merge(left=chances, right=seats, on='vars', how='inner').rename({'vars': 'Party'}, axis=1)
summary_stats.to_csv('display_data/summary_stats.csv')
summary_stats

,Party,Win Probability,Seat Share
0,Democrats,65.1875,52.197875
1,Republicans,34.8125,47.802125


In [303]:
incl_cols = ['state', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'y_pred', 'flip_indic', 'matchup', 'swing_24_to_26', 'disp_24_to_26_swing']
disp_data = data[incl_cols]

In [304]:
rating_ts = output_ts[output_ts['type'] == 'chance']
rating_ts['rating'] = rating_ts['y'].map(get_rating)
rating_ts['date'] = pd.to_datetime(rating_ts['date'])
rating_ts.head()

,date,y,geo,type,rating
1,2026-09-27,58.6000,US Senate,chance,Tossup
5,2026-09-27,0.0500,AL,chance,Safe R
10,2026-09-27,79.8875,AK,chance,Likely D
15,2026-09-27,0.1250,AR,chance,Safe R
20,2026-09-27,99.9625,CO,chance,Safe D


In [305]:
## Alert when there's a rating change
today_date = rating_ts['date'].max()
yesterday = rating_ts[rating_ts['date'] != today_date]['date'].max()

today_ratings = rating_ts[rating_ts['date'] == today_date]
yester_ratings = rating_ts[rating_ts['date'] == yesterday]
compare_df = pd.merge(left=today_ratings, right=yester_ratings, on='geo', how='left', suffixes=['_t', '_y'])
compare_df.head()

,date_t,y_t,geo,type_t,rating_t,date_y,y_y,type_y,rating_y
0,2026-11-02,65.1875,US Senate,chance,Lean D,2026-11-01,65.0000,chance,Lean D
1,2026-11-02,0.6000,AL,chance,Safe R,2026-11-01,0.5750,chance,Safe R
2,2026-11-02,82.4875,AK,chance,Likely D,2026-11-01,83.4000,chance,Likely D
3,2026-11-02,0.2125,AR,chance,Safe R,2026-11-01,0.1875,chance,Safe R
4,2026-11-02,99.9250,CO,chance,Safe D,2026-11-01,99.9500,chance,Safe D


In [306]:
## This is the cell that actually alerts when there's a rating change
if any(compare_df['rating_t'] != compare_df['rating_y']):
    changes_df = compare_df[compare_df['rating_t'] != compare_df['rating_y']][['geo', 'rating_y', 'rating_t']]
    print(changes_df)
else:
    print("No rating changes today!")

No rating changes today!


In [307]:
for i in range(changes_df.shape[0]):
    seat = changes_df.iloc[i, :]['geo']
    rating_y = changes_df.iloc[i, :]['rating_y']
    rating_t = changes_df.iloc[i, :]['rating_t']
    print(f'{seat}: {rating_y} --> {rating_t}')

US Senate: Tilt D --> Lean D
OH: Tossup --> Tilt D
TX: Tilt D --> Lean D


In [308]:
if any(compare_df['rating_t'] != compare_df['rating_y']):
    print(changes_df.shape)

In [309]:
## Mini-widget: what is the chance of Dems/Reps winning Senate if X state is won by Dems/Reps?
def det_winner(post, state, dem_winner=True):
    '''
    :param post: Posterior simulation matrix.
    :param state: Congressional district.
    :param dem_winner: Whether or not we set Dems or Reps to win `state`. True if Dem win is fixed, False if Rep win is fixed.
    '''
    winners = []
    seats = []
    post_ut = post.T
    if dem_winner:
        post_ut = post_ut[post_ut[state] > 0]
    else:
        post_ut = post_ut[post_ut[state] < 0]
    post_case = post_ut.T
    for i in range(post_case.shape[0]):
        sim = post_case.iloc[:, i]
        dem_seats = np.sum(sim > 0)
        if dem_seats >= 51 - dem_uncont: # Democrats win House in this draw
            winners.append(1)
        else: # Republicans win House in this draw
            winners.append(0)
        seats.append(dem_seats)
    dem_win_chance = np.mean(np.array(winners)) * 100
    dem_mean_seats = np.mean(np.array(seats)) + dem_uncont
    return dem_win_chance, dem_mean_seats

st = 'TX'
dem_winner = True
cond_win_chance, cond_seats = det_winner(post, st, dem_winner)
print(f'If Dems {'win' if dem_winner else 'lose'} {st}-SEN, they have a {cond_win_chance:.1f}% chance of winning the chamber overall, with mean projected seats of {cond_seats:.1f}')

If Dems win TX-SEN, they have a 80.0% chance of winning the chamber overall, with mean projected seats of 53.0


In [310]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
tab_data.to_csv('display_data/table_display_data.csv')
data.to_csv('display_data/all_data.csv')